# 5. Classification and the Y3 Hurdle Model

Point regression has now been measured across five model families, two split geometries
and two feature counts. On Y1 nothing beats a constant. Rather than tune harder against
that wall, this stage asks a strictly easier question of the same data: not *how large*
the move was, but *which side of a line* it fell on. A binary outcome needs far less
information than a point estimate, which is what makes it the more tractable framing at
N = 64.

It is also where precision, recall, F1 and AUC genuinely live. The earlier notebook
obtained them by thresholding a regressor's output after the fact; that is a diagnostic,
not a classifier, and it is why the best directional accuracy landed exactly on its own
majority baseline.

### Every threshold is fixed before any model is fitted

| Label | Definition | Where the threshold comes from |
|---|---|---|
| `C1_negative_return` | `Y1 < 0` | The sign split. No free parameter. |
| `C1b_adverse_move` | `Y1` below the bottom tercile of **that fold's training window** | Computed on training rows only. The cut point moves per fold; the *rule* is fixed. It gives ~33% prevalence in the **training** window by construction, but test-fold prevalence can be much lower — the held-out folds are calmer than the training folds (Y1 test sigma is 0.64x full-sample), so fewer test events fall below a cut set on turbulent training data. |
| `C2_volume_spike` | `Y2 > 0` | Y2 is defined as `V/V̄ − 1`, so zero is where the target is centred by construction. |
| `C3_recovers_in_90` | `Y3 < 90` | The design constant itself. This is also stage 1 of the hurdle model. |

**A threshold rejected in advance.** `Y1 < −1σ` (one pre-event standard deviation, the
standardised-abnormal-return form) yields roughly 7–8 positives across all 64 events —
about 3 in the pooled test set. Precision and recall on 3 positives are not estimable, so
it is reported as a sensitivity check and never as a headline.

### How a classifier is credited with working

Accuracy is gameable by the majority rule, which is exactly how the earlier 65%-vs-65%
result arose. So the headline metrics here are **balanced accuracy** and **Matthews
correlation**, for which the always-predict-majority classifier scores exactly 0.5 and
0.0 respectively. A classifier counts as working only if it clears the majority rule on a
baseline-proof metric **and** its AUC interval excludes 0.5.

## 5.1 Environment and cached inputs

Loads `_shared.py` (paths, the artifact cache helpers, the target definitions) and applies the thesis figure style, then prints the artifact cache so it is visible which upstream stage produced these inputs and when. Reads `dataset` and the same `splits` stage 04 used, so the two stages are directly
comparable.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "_shared.py").exists() else Path.cwd() / "notebooks"))
from _shared import *  # noqa: F401,F403

import numpy as np
from src.evaluation import figures as fx
fx.apply_thesis_style()

dataset = load_frame("dataset")
spec = load_json("feature_spec")
FEATURE_COLS, TARGET_COLS = spec["FEATURE_COLS"], spec["TARGET_COLS"]
X = dataset[FEATURE_COLS].fillna(0.0)
y = dataset[TARGET_COLS].copy()
splits = load_object("splits")
print(f"{len(dataset)} events x {len(FEATURE_COLS)} features, {len(splits)} folds")


76 events x 63 features, 4 folds


## 5.2 The walk-forward classification loop

Identical geometry, identical per-fold feature selection and identical inner
`TimeSeriesSplit` hyperparameter search as the regression stage, so the two are directly
comparable.

One deliberate difference: **no SMOGN.** The augmentation interpolates between continuous
target values, which has no meaning for a class label. Imbalance is handled instead by
`class_weight="balanced"`, fixed a priori from prevalence and never tuned.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from src.models.classifiers import LABELS, build_classifiers, classification_metrics, summarise


def select_top_features(X_tr, labels_tr, k=20, random_state=RANDOM_STATE):
    """RF-importance top-K, fitted on the fold's training rows only."""
    k = min(k, X_tr.shape[1])
    probe = RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=3,
                                   random_state=random_state, n_jobs=-1)
    probe.fit(X_tr, labels_tr)
    order = np.argsort(probe.feature_importances_)[::-1][:k]
    return list(X_tr.columns[order])


def inner_cv(n_rows, n_splits=3):
    return TimeSeriesSplit(n_splits=max(2, min(n_splits, n_rows - 1)))


def run_classification(splits, X_all, y_all, k_features=20):
    store = {name: {m: {"y_true": [], "y_score": []} for m in build_classifiers()}
             for name in LABELS}

    for name, label_fn in LABELS.items():
        for fold_i, s in enumerate(splits):
            tr, te = np.asarray(s.train_index), np.asarray(s.test_index)
            labels = label_fn(y_all, tr, dataset)          # cut point from training rows only

            ok_tr = tr[labels.iloc[tr].notna().to_numpy()]
            ok_te = te[labels.iloc[te].notna().to_numpy()]
            l_tr = labels.iloc[ok_tr].to_numpy().astype(int)
            l_te = labels.iloc[ok_te].to_numpy().astype(int)
            if len(np.unique(l_tr)) < 2 or len(ok_te) == 0:
                continue                                    # nothing to learn from this fold

            X_tr, X_te = X_all.iloc[ok_tr], X_all.iloc[ok_te]
            feats = select_top_features(X_tr, l_tr, k=k_features)
            X_tr, X_te = X_tr[feats], X_te[feats]

            for model_name, (estimator, grid) in build_classifiers().items():
                # error_score=np.nan keeps a single-class inner split from aborting the
                # whole search: with C3 at ~86% prevalence and inner blocks of 5-15 rows,
                # some validation splits contain no negative case at all, and roc_auc is
                # undefined there. Those splits score NaN and the rest still decide.
                search = GridSearchCV(estimator, grid, cv=inner_cv(len(X_tr)),
                                      scoring="roc_auc", n_jobs=-1, refit=True,
                                      error_score=np.nan)
                try:
                    search.fit(X_tr, l_tr)
                    score = search.best_estimator_.predict_proba(X_te)[:, 1]
                except Exception:
                    # Every inner split degenerated. Fall back to the estimator at its
                    # declared defaults rather than dropping the fold silently.
                    try:
                        fitted = clone(estimator).fit(X_tr, l_tr)
                        score = fitted.predict_proba(X_te)[:, 1]
                    except Exception:
                        score = np.full(len(X_te), l_tr.mean())
                store[name][model_name]["y_true"].append(l_te)
                store[name][model_name]["y_score"].append(score)
    return store


clf_store = run_classification(splits, X, y)
print("classification folds complete")


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
  warnings.warn(


...

C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:1103: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
  warnings.warn(


classification folds complete


## 5.3 Results

Read the `beats_baseline` column first. Everything to its left describes the classifier;
that column says whether the description amounts to anything.

In [3]:
records = []
for label_name, per_model in clf_store.items():
    for model_name, arrays in per_model.items():
        if not arrays["y_true"]:
            continue
        yt = np.concatenate(arrays["y_true"])
        ys = np.concatenate(arrays["y_score"])
        rec = classification_metrics(yt, ys)
        rec.update({"label": label_name, "model": model_name})
        records.append(rec)

        # The majority rule on the same pooled test points, as the yardstick.
        maj = classification_metrics(yt, np.full(len(yt), float(yt.mean() >= 0.5)))
        maj.update({"label": label_name, "model": "majority_baseline"})
        records.append(maj)

clf_summary = summarise(records).drop_duplicates(subset=["label", "model"])
pd.set_option("display.width", 220)
print(clf_summary.to_string(index=False))
save_frame(clf_summary, "classification_summary", "Pre-registered binary labels, pooled out-of-fold.")


             label             model  n  n_pos  prevalence  majority_baseline_acc  accuracy  balanced_accuracy       mcc  precision   recall       f1   pr_auc      auc   auc_lo   auc_hi  auc_boot_lo  auc_boot_hi  beats_baseline
C1_negative_return          logistic 40     13    0.325000               0.675000  0.575000           0.505698  0.011648   0.333333 0.307692 0.320000 0.397891 0.478632 0.286411 0.670854     0.273504     0.686610           False
C1_negative_return majority_baseline 40     13    0.325000               0.675000  0.675000           0.500000  0.000000   0.000000 0.000000 0.000000 0.325000 0.500000 0.306623 0.693377     0.500000     0.500000           False
C1_negative_return            rf_clf 40     13    0.325000               0.675000  0.550000           0.487179 -0.025641   0.307692 0.307692 0.307692 0.353311 0.501425 0.307986 0.694863     0.307692     0.698006           False
C1_negative_return           xgb_clf 40     13    0.325000               0.675000  0.600

cached classification_summary.parquet  (24 rows x 19 cols)


WindowsPath('F:/CSE-disaster-impact-predictor/disaster_finance_predictor/artifacts/classification_summary.parquet')

### 5.3.1 Verdict per label

Each label is scored against the majority rule on the same rows. A label whose best
model cannot beat that rule on balanced accuracy, with an AUC interval excluding 0.5,
is reported as a failure rather than as its raw accuracy.

In [4]:
print("CLASSIFICATION VERDICT\n" + "=" * 72)
for label_name in LABELS:
    sub = clf_summary[(clf_summary["label"] == label_name)
                      & (clf_summary["model"] != "majority_baseline")]
    if sub.empty:
        print(f"{label_name}: no fold had both classes present.")
        continue
    winners = sub.loc[sub["beats_baseline"], "model"].tolist()
    best = sub.loc[sub["auc"].idxmax()]
    if winners:
        print(f"{label_name}: WORKS -- {', '.join(winners)} clear the majority rule "
              f"with an AUC interval excluding 0.5.")
    else:
        print(f"{label_name}: not distinguishable from chance. Best AUC {best['auc']:.3f} "
              f"[{best['auc_lo']:.2f}, {best['auc_hi']:.2f}] ({best['model']}), "
              f"prevalence {best['prevalence']:.2f}, "
              f"balanced accuracy {best['balanced_accuracy']:.3f} vs 0.500 for the majority rule.")


CLASSIFICATION VERDICT
C1_negative_return: not distinguishable from chance. Best AUC 0.501 [0.31, 0.69] (rf_clf), prevalence 0.33, balanced accuracy 0.487 vs 0.500 for the majority rule.
C1b_adverse_move: not distinguishable from chance. Best AUC 0.608 [0.35, 0.87] (logistic), prevalence 0.15, balanced accuracy 0.495 vs 0.500 for the majority rule.
C2_volume_spike: WORKS -- logistic, rf_clf, xgb_clf clear the majority rule with an AUC interval excluding 0.5.
C3_recovers_in_90: not distinguishable from chance. Best AUC 0.618 [0.35, 0.89] (logistic), prevalence 0.90, balanced accuracy 0.472 vs 0.500 for the majority rule.
C3b_slow_recovery: not distinguishable from chance. Best AUC 0.510 [0.32, 0.70] (rf_clf), prevalence 0.33, balanced accuracy 0.486 vs 0.500 for the majority rule.
C4_car5_negative: WORKS -- logistic clear the majority rule with an AUC interval excluding 0.5.


## 5.4 The Y3 hurdle model

Y3 is a point mass at 0, a short right tail and a second point mass at the 90-day cap.
A single continuous regressor across that shape is misspecified, which is the most likely
reason the training-mean predictor beats every fitted model on this target.

The hurdle matches the structure: stage 1 predicts the probability of recovering inside
the window, stage 2 regresses `log1p(duration)` on the events that actually recovered, and
the two combine as `P·E[d | recovered] + (1 − P)·90`. Stage 2 never sees a censored 90, so
the cap stops being treated as an observed duration.

**Disclose alongside any result from this:** stage 1 is close to a sign classifier for Y1,
because `Y3 = 0` if and only if `Y1 ≥ 0`. The comparison that decides it is MAE in trading
days against `naive_zero`, which is the tighter of the two nulls on that metric.

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

from src.evaluation.metrics import clip_to_bounds
from src.models.hurdle import HurdleRecoveryModel

T3 = "Y3_recovery_days"
hurdle_true, hurdle_pred, single_pred, zero_pred, mean_pred = [], [], [], [], []

for s in splits:
    tr, te = np.asarray(s.train_index), np.asarray(s.test_index)
    ok_tr = tr[y[T3].iloc[tr].notna().to_numpy()]
    ok_te = te[y[T3].iloc[te].notna().to_numpy()]
    y_tr, y_te = y[T3].iloc[ok_tr].to_numpy(), y[T3].iloc[ok_te].to_numpy()

    feats = select_top_features(X.iloc[ok_tr], (y_tr < 90).astype(int))
    X_tr, X_te = X.iloc[ok_tr][feats], X.iloc[ok_te][feats]

    hurdle = HurdleRecoveryModel(
        LogisticRegression(max_iter=5000, class_weight="balanced"),
        RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=3,
                              random_state=RANDOM_STATE, n_jobs=-1),
    ).fit(X_tr, y_tr)

    single = RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=3,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    single.fit(X_tr, np.log1p(y_tr))

    hurdle_true.append(y_te)
    hurdle_pred.append(hurdle.predict(X_te))
    single_pred.append(clip_to_bounds(T3, np.expm1(single.predict(X_te))))
    zero_pred.append(np.zeros_like(y_te))
    mean_pred.append(np.full_like(y_te, float(np.mean(y_tr))))

yt = np.concatenate(hurdle_true)
rows = []
for nm, pr in (("hurdle", hurdle_pred), ("single-stage RF", single_pred),
               ("naive_zero", zero_pred), ("naive_train_mean", mean_pred)):
    p = np.concatenate(pr)
    rows.append({"model": nm, "n": len(yt),
                 "MAE_trading_days": float(np.mean(np.abs(p - yt))),
                 "RMSE": float(np.sqrt(np.mean((p - yt) ** 2)))})
hurdle_table = pd.DataFrame(rows).sort_values("MAE_trading_days")
print(hurdle_table.to_string(index=False))


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


           model  n  MAE_trading_days      RMSE
      naive_zero 40         13.725000 32.254070
 single-stage RF 40         14.374109 32.034311
naive_train_mean 40         19.921667 29.468642
          hurdle 40         24.145869 41.155945


### 5.4.1 Does the hurdle model beat the naive baseline?

Paired event-level bootstrap and Diebold-Mariano on MAE, against the constant-zero
null. Both must agree before the improvement counts.

In [6]:
from src.evaluation.verification import diebold_mariano, paired_bootstrap_delta

base = np.concatenate(zero_pred)
h = np.concatenate(hurdle_pred)
boot = paired_bootstrap_delta(yt, h, base, metric="mae")
dm = diebold_mariano(yt, h, base, power=1)
print(f"\nHurdle vs naive_zero on MAE: delta = {boot['delta']:+.3f} trading days, "
      f"95% CI [{boot['ci_low']:+.3f}, {boot['ci_high']:+.3f}], DM p = {dm['p_value']:.3f}")
if boot["significant"] and dm["significant"]:
    verdict_txt = "hurdle beats the null"
elif boot["ci_high"] < 0:
    # The interval lies entirely on the wrong side of zero: this is not an inconclusive
    # result, it is a decisive negative, and saying "not distinguishable" would understate it.
    verdict_txt = ("hurdle is SIGNIFICANTLY WORSE than the null -- rejected by its own "
                   "pre-registered criterion; the single-stage model stays primary")
else:
    verdict_txt = ("not distinguishable from the null -- report as tried and not better, "
                   "and keep the simpler single-stage model as primary")
print("VERDICT:", verdict_txt)
print("Why it fails: Y3 has median 0, so the null that predicts 0 is already strong")
print("on MAE. The hurdle always carries a (1 - P(recover)) * 90 term, which puts a")
print("floor under every prediction -- ruinous against a target whose typical value")
print("is zero.")

save_object({"classification": clf_store,
             "hurdle": {"y_true": hurdle_true, "hurdle": hurdle_pred,
                        "single": single_pred, "zero": zero_pred, "mean": mean_pred}},
            "results_classification",
            "Pre-registered binary labels and the Y3 hurdle model, pooled out-of-fold.")
save_frame(hurdle_table, "hurdle_table", "Y3 hurdle vs single-stage vs both nulls.")



Hurdle vs naive_zero on MAE: delta = -10.421 trading days, 95% CI [-19.668, -2.487], DM p = 0.022
VERDICT: hurdle is SIGNIFICANTLY WORSE than the null -- rejected by its own pre-registered criterion; the single-stage model stays primary
Why it fails: Y3 has median 0, so the null that predicts 0 is already strong
on MAE. The hurdle always carries a (1 - P(recover)) * 90 term, which puts a
floor under every prediction -- ruinous against a target whose typical value
is zero.


cached results_classification.pkl


cached hurdle_table.parquet  (4 rows x 4 cols)


WindowsPath('F:/CSE-disaster-impact-predictor/disaster_finance_predictor/artifacts/hurdle_table.parquet')